<a href="https://colab.research.google.com/github/wmde/WDumps-scraper/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install requests-cache

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00


In [ ]:
from bs4 import BeautifulSoup
import requests
import csv
import requests_cache

In [23]:
def get_cache(url):
  session = requests_cache.CachedSession('dump_cache')
  response = session.get(url)
  if response.status_code == 200:
    return response.text
  else:
     return False

def get_temp_cache(url):
  session = requests_cache.CachedSession('temp_cache', expire_after=7200)
  response = session.get(url)
  if response.status_code == 200:
    return response.text
  else:
     return False

def get_name(html_content):
  soup = BeautifulSoup(html_content, "html.parser")
  dump_header = soup.find("h2")
  if dump_header is not None:
    name = dump_header.text.split(": ", maxsplit=1)[-1]
    return name
  else:
    return ""

# Extract latest dump ID

In [ ]:
start = "https://wdumps.toolforge.org/dumps"
html = get(start)
soup = BeautifulSoup(html, "html.parser")

link_elem = soup.find("table").find("a")
link_url = link_elem["href"]
last_id = int(link_url.split("/")[-1])
print(last_id)

5410


# Extract IDs and names
Save these in a dictionary with the URLs.

In [27]:
base_url = "https://wdumps.toolforge.org/dump/"
dump = {}
dumps = []

for i in range(1, last_id + 1):
  url = f"{base_url}{i}"
  if i > 5400:
    html = get_temp_cache(url)
  else:
    html = get_cache(url)
  if html:
    dump["url"] = url
    dump["id"] = i
    dump["name"] = get_name(html)
    dumps.append(dump.copy())

# Save to csv

In [28]:
with open("POC-Wikidata-dumps.csv", "w", newline="") as csvfile:
  fieldnames = ["url", "id", "name"]
  writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
  writer.writeheader()

  for i in range(0, len(dumps)):
    writer.writerow({"url": dumps[i]["url"], "id": dumps[i]["id"], "name": dumps[i]["name"]})